In [40]:
# ============================================================
# Hybrid Rocket Fuel Blend Analysis
# Inputs  : X_Value, Force, Pressure_0, Gox
# Outputs : Pressure, rdot_mm_s, Isp, Thrust, cstar
# LightGBM multi-output + MC Dropout (unchanged)
# Exports: Best_Fuel_Results_LGBM.xlsx + Best_Fuel_Summary_LGBM.txt
#          Gox_vs_Predicted_Regression_Rate_LGBM.png
# ============================================================

from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam

from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore", category=FutureWarning)

In [41]:
# ============================================================
# 1. CONFIG
# ============================================================

EXCEL_PATH = Path("daq_raw_4blends.xlsx")
OUTPUT_EXCEL = Path("Best_Fuel_Results_LGBM.xlsx")
OUTPUT_TXT = Path("Best_Fuel_Summary_LGBM.txt")
OUTPUT_GRAPH = Path("Gox_vs_Predicted_Regression_Rate_LGBM.png")
OUTPUT_GRAPH1 = Path("Pressure_vs_Predicted_Regression_Rate_LGBM.png")

SHEETS_TO_USE = None
REQUIRED_SHEET_COUNT = 4
FUEL_COLUMN = "fuel_type"

FEATURE_COLUMNS = ["X_Value", "Force", "Pressure_0", "Gox"]

TARGET_COLUMNS = ["Pressure", "rdot_mm_s", "Isp", "Thrust", "cstar"]

TARGET_WEIGHTS = {t: 1.0 for t in TARGET_COLUMNS}

RDOT_TARGET = "rdot_mm_s"
PRED_RDOT = f"pred_{RDOT_TARGET}"

SEED = 100
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20
N_MC_SAMPLES = 100
UNCERTAINTY_PENALTY_K = 0.5
MIN_LABELED_ROWS = 20

# LightGBM hyperparameters
N_ESTIMATORS = 300
MAX_DEPTH = 3
LEARNING_RATE = 0.05
NUM_LEAVES = 8
MIN_CHILD_SAMPLES = 10
SUBSAMPLE = 0.8
COLSAMPLE = 0.8
REG_ALPHA = 1.0
REG_LAMBDA = 5.0

np.random.seed(SEED)
tf.random.set_seed(SEED)

In [42]:
# ============================================================
# 2. HELPERS
# ============================================================

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def clean_numeric(df, columns):
    out = df.copy()
    for c in columns:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
        else:
            out[c] = np.nan
    return out


def safe_r2_multi(y_true, y_pred):
    scores = []
    for i in range(y_true.shape[1]):
        if np.unique(y_true[:, i]).size < 2:
            scores.append(np.nan)
        else:
            scores.append(r2_score(y_true[:, i], y_pred[:, i]))
    return np.asarray(scores, dtype=float)


def minmax_norm(series):
    low, high = series.min(), series.max()
    if not np.isfinite(low) or not np.isfinite(high) or abs(high - low) < 1e-12:
        return pd.Series(0.0, index=series.index)
    return (series - low) / (high - low)


def mc_dropout_predict(model, x_input, n_samples):
    preds = np.empty((n_samples, len(x_input)), dtype=np.float32)
    for i in range(n_samples):
        preds[i] = model(x_input, training=True).numpy().ravel()
    return preds.mean(axis=0), preds.std(axis=0)


def build_lgbm_pipeline(seed, feature_cols):
    """LightGBM multi-output pipeline."""
    preprocessor = ColumnTransformer(
        transformers=[
            ("fuel", make_one_hot_encoder(), [FUEL_COLUMN]),
            ("numeric", "passthrough", feature_cols),
        ],
        remainder="drop",
    )
    base = LGBMRegressor(
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        learning_rate=LEARNING_RATE,
        num_leaves=NUM_LEAVES,
        min_child_samples=MIN_CHILD_SAMPLES,
        subsample=SUBSAMPLE,
        colsample_bytree=COLSAMPLE,
        reg_alpha=REG_ALPHA,
        reg_lambda=REG_LAMBDA,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )
    return Pipeline([("preprocessor", preprocessor), ("model", MultiOutputRegressor(base))])


def validate_excel_file(path: Path):
    if not path.exists():
        raise FileNotFoundError(
            f"Excel not found: {path.resolve()}\n"
            "Put daq_raw_4blends.xlsx in the same folder as this script/notebook."
        )
    size = path.stat().st_size
    if size < 100:
        raise ValueError(f"File is too small ({size} bytes) to be a valid Excel workbook: {path}")
    with open(path, "rb") as f:
        magic = f.read(4)
    if magic[:2] != b"PK":
        raise ValueError(
            f"File is not a valid .xlsx (zip) file: {path.resolve()}\n"
            f"First bytes: {magic!r}\n"
            "Open in Excel and Save As → Excel Workbook (.xlsx), then re-upload."
        )


In [43]:
# ============================================================
# 3. LOAD
# ============================================================

def load_dataset():
    validate_excel_file(EXCEL_PATH)

    try:
        xls = pd.ExcelFile(EXCEL_PATH, engine="openpyxl")
    except Exception as exc:
        raise ValueError(
            f"Could not open Excel with openpyxl: {EXCEL_PATH.resolve()}\n"
            f"Original error: {exc}"
        ) from exc

    sheet_names = xls.sheet_names if SHEETS_TO_USE is None else list(SHEETS_TO_USE)

    if len(sheet_names) < REQUIRED_SHEET_COUNT:
        raise ValueError(
            f"Need >= {REQUIRED_SHEET_COUNT} sheets; found {len(sheet_names)}. "
            f"Sheets present: {xls.sheet_names}"
        )

    sheet_names = sheet_names[:REQUIRED_SHEET_COUNT]
    frames = []
    gox_col_global = "Gox" if "Gox" in FEATURE_COLUMNS else None
    pressure_col_global = "Pressure" if "Pressure" in TARGET_COLUMNS else None

    for sheet in sheet_names:
        raw = pd.read_excel(xls, sheet_name=sheet, engine="openpyxl")

        missing_in = [c for c in FEATURE_COLUMNS if c not in raw.columns]
        if missing_in:
            raise ValueError(
                f"Sheet '{sheet}' missing INPUT columns: {missing_in}\n"
                f"Columns present: {list(raw.columns)}"
            )

        keep = list(FEATURE_COLUMNS)
        for t in TARGET_COLUMNS:
            if t not in raw.columns:
                raw[t] = np.nan
            if t not in keep:
                keep.append(t)

        keep_unique = list(dict.fromkeys(keep))
        frame = clean_numeric(raw[keep_unique], keep_unique)
        frame[FUEL_COLUMN] = str(sheet)
        frames.append(frame)

    data = pd.concat(frames, ignore_index=True)
    data = data.dropna(subset=FEATURE_COLUMNS).reset_index(drop=True)
    if "X_Value" in data.columns:
        data = data[data["X_Value"] >= 0].reset_index(drop=True)

    if data.empty:
        raise ValueError("No rows with valid INPUT columns after cleaning.")

    labeled_mask = data[TARGET_COLUMNS].notna().all(axis=1)
    for t in TARGET_COLUMNS:
        labeled_mask &= np.isfinite(pd.to_numeric(data[t], errors="coerce"))

    n_labeled = int(labeled_mask.sum())

    print(f"Sheets used      : {sheet_names}")
    print(f"Total input rows : {len(data)}")
    print(f"Labeled rows     : {n_labeled}  (all targets filled — used for training)")
    print(f"Unlabeled rows   : {len(data) - n_labeled}  (inputs only — prediction only)")
    print(f"Gox column       : {gox_col_global}")
    print(f"Targets          : {TARGET_COLUMNS}")
    print(f"Pred rdot column : {PRED_RDOT}")
    print(f"Main model       : LightGBM")

    if n_labeled < MIN_LABELED_ROWS:
        raise ValueError(
            f"Need at least {MIN_LABELED_ROWS} rows with ALL target columns filled to train. "
            f"Found {n_labeled}.\n"
            "Fill Pressure, rdot_mm_s, Isp, Thrust, cstar for more rows, "
            "or lower MIN_LABELED_ROWS."
        )

    return data, sheet_names, labeled_mask, gox_col_global, pressure_col_global



In [44]:
# ============================================================
# 4. MC DROPOUT  (unchanged)
# ============================================================

def build_mc_dropout_inputs(x_train, x_other, feature_cols):
    encoder = make_one_hot_encoder()
    train_fuel = encoder.fit_transform(x_train[[FUEL_COLUMN]])
    other_fuel = encoder.transform(x_other[[FUEL_COLUMN]])

    scaler = StandardScaler()
    train_num = scaler.fit_transform(x_train[feature_cols]).astype(np.float32)
    other_num = scaler.transform(x_other[feature_cols]).astype(np.float32)

    return (
        np.hstack([train_num, train_fuel]).astype(np.float32),
        np.hstack([other_num, other_fuel]).astype(np.float32),
        encoder,
        scaler,
    )


def train_mc_dropout_network(x_train_nn, y_train_rdot):
    x_fit, x_val, y_fit, y_val = train_test_split(
        x_train_nn, y_train_rdot, test_size=VALIDATION_SIZE, random_state=SEED
    )

    model = Sequential([
        Input(shape=(x_train_nn.shape[1],)),
        Dense(64, activation="relu"),
        Dropout(0.30),
        Dense(32, activation="relu"),
        Dropout(0.30),
        Dense(16, activation="relu"),
        Dropout(0.20),
        Dense(1),
    ])
    model.compile(optimizer=Adam(learning_rate=0.002), loss="mse")
    es = EarlyStopping(
        monitor="val_loss", patience=15, restore_best_weights=True, verbose=0
    )
    model.fit(
        x_fit, y_fit,
        validation_data=(x_val, y_val),
        epochs=200, batch_size=64, callbacks=[es], verbose=0,
    )
    return model


In [45]:
# ============================================================
# 5. PLOT
# ============================================================

def plot_gox_vs_rdot(results, gox_col, sheet_names):
    if gox_col is None or gox_col not in results.columns:
        print("Skipping Gox vs rdot plot: no Gox column in data.")
        return
    if PRED_RDOT not in results.columns:
        print(f"Skipping plot: {PRED_RDOT} not in results.")
        return

    plot_df = results[[FUEL_COLUMN, gox_col, PRED_RDOT]].dropna()
    plot_df = plot_df[
        np.isfinite(plot_df[gox_col]) & np.isfinite(plot_df[PRED_RDOT])
    ]
    if plot_df.empty:
        print("Skipping Gox vs rdot plot: no finite pairs.")
        return

    fig, ax = plt.subplots(figsize=(9, 6))
    for fuel in sheet_names:
        sub = plot_df[plot_df[FUEL_COLUMN] == fuel]
        if sub.empty:
            continue
        ax.plot(sub[gox_col], sub[PRED_RDOT], marker="o", markersize=4, linewidth=1.8, alpha=0.8, label=str(fuel))

    ax.set_xlabel("Oxidizer mass flux  Gox")
    ax.set_ylabel("Predicted regression rate  (mm/s)")
    ax.set_title("Gox vs Predicted Regression Rate (LightGBM)")
    ax.grid(True, alpha=0.3)
    ax.legend(title="Fuel blend", fontsize=8)
    fig.tight_layout()
    fig.savefig(OUTPUT_GRAPH, dpi=150)
    plt.close(fig)
    print(f"Graph saved: {OUTPUT_GRAPH.resolve()}")


In [46]:
# ============================================================
# 6. PLOT: PREDICTED PRESSURE VS PREDICTED REGRESSION RATE
# ============================================================

def plot_pressure_vs_rdot(results, pressure_col, sheet_names):

    if pressure_col is None or pressure_col not in results.columns:
        print("Skipping plot: Predicted pressure column not found.")
        return

    if PRED_RDOT not in results.columns:
        print(f"Skipping plot: {PRED_RDOT} not found.")
        return

    plot_df = results[[FUEL_COLUMN, pressure_col, PRED_RDOT]].dropna()

    plot_df = plot_df[np.isfinite(plot_df[pressure_col]) & np.isfinite(plot_df[PRED_RDOT])]

    if plot_df.empty:
        print("Skipping plot: No valid pressure and regression rate data.")
        return

    fig, ax = plt.subplots(figsize=(9, 6))

    for fuel in sheet_names:
        sub = plot_df[plot_df[FUEL_COLUMN] == fuel]

        if sub.empty:
            continue

        ax.plot(sub[pressure_col], sub[PRED_RDOT], marker="o", markersize=4, linewidth=1.8, alpha=0.8, label=str(fuel))

    ax.set_xlabel("Predicted Pressure")
    ax.set_ylabel("Predicted Regression Rate (mm/s)")
    ax.set_title("Predicted Pressure vs Predicted Regression Rate")

    ax.grid(True, alpha=0.25)
    ax.legend(title="Fuel Blend", fontsize=8)

    fig.tight_layout()
    fig.savefig(OUTPUT_GRAPH1, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"Graph saved: {OUTPUT_GRAPH1.resolve()}")

In [47]:
# ============================================================
# 7. MAIN
# ============================================================

def main():
    if RDOT_TARGET not in TARGET_COLUMNS:
        raise ValueError(f"{RDOT_TARGET} must be in TARGET_COLUMNS={TARGET_COLUMNS}")

    data, sheet_names, labeled_mask, gox_col, pressure_col = load_dataset()
    feature_cols = list(FEATURE_COLUMNS)

    labeled = data.loc[labeled_mask].reset_index(drop=True)
    x_lab = labeled[feature_cols + [FUEL_COLUMN]].copy()
    y_lab = labeled[TARGET_COLUMNS].to_numpy(dtype=float)

    counts = labeled[FUEL_COLUMN].value_counts()
    stratify = labeled[FUEL_COLUMN] if counts.min() >= 2 else None

    x_train, x_test, y_train, y_test = train_test_split(
        x_lab, y_lab, test_size=TEST_SIZE, random_state=SEED, stratify=stratify
    )
    print(f"Train (labeled): {len(x_train)} | Test (labeled): {len(x_test)}")

    print("\nTraining Multi-Output LightGBM on labeled data...")
    lgbm_model = build_lgbm_pipeline(SEED, feature_cols).fit(x_train, y_train)

    train_pred = lgbm_model.predict(x_train)
    test_pred = lgbm_model.predict(x_test)
    train_r2 = safe_r2_multi(y_train, train_pred)
    test_r2 = safe_r2_multi(y_test, test_pred)
    test_mae = mean_absolute_error(y_test, test_pred, multioutput="raw_values")
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred, multioutput="raw_values"))

    print("\nPer-target performance (labeled test set) — LightGBM:")
    for i, t in enumerate(TARGET_COLUMNS):
        gap = (
            train_r2[i] - test_r2[i]
            if np.isfinite(train_r2[i]) and np.isfinite(test_r2[i])
            else np.nan
        )
        print(
            f"  {t:<20} Train R2={train_r2[i]:.3f} | Test R2={test_r2[i]:.3f} "
            f"| gap={gap:.3f} | MAE={test_mae[i]:.4f} | RMSE={test_rmse[i]:.4f}"
        )

    x_all = data[feature_cols + [FUEL_COLUMN]].copy()
    all_pred = lgbm_model.predict(x_all)

    results = data.copy()
    for i, t in enumerate(TARGET_COLUMNS):
        results[f"pred_{t}"] = all_pred[:, i]

    # ----- MC Dropout (unchanged) -----
    rdot_idx = TARGET_COLUMNS.index(RDOT_TARGET)
    print("\nTraining MC Dropout network (val from train only)...")
    train_nn, all_nn, _, _ = build_mc_dropout_inputs(x_train, x_all, feature_cols)
    _, test_nn, _, _ = build_mc_dropout_inputs(x_train, x_test, feature_cols)

    nn = train_mc_dropout_network(train_nn, y_train[:, rdot_idx].astype(np.float32))

    mc_mean_test, mc_std_test = mc_dropout_predict(nn, test_nn, N_MC_SAMPLES)
    y_test_rdot = y_test[:, rdot_idx]
    mc_r2 = (
        r2_score(y_test_rdot, mc_mean_test)
        if np.unique(y_test_rdot).size > 1
        else np.nan
    )
    mc_mae = mean_absolute_error(y_test_rdot, mc_mean_test)
    print(
        f"MC Dropout TEST R2={mc_r2:.3f} | MAE={mc_mae:.4f} | "
        f"mean sigma={mc_std_test.mean():.4f}"
    )

    _, mc_std_all = mc_dropout_predict(nn, all_nn, N_MC_SAMPLES)
    results["uncertainty_rdot"] = mc_std_all

    # Rank on labeled TEST only
    test_results = x_test.copy().reset_index(drop=True)
    for i, t in enumerate(TARGET_COLUMNS):
        test_results[t] = y_test[:, i]
        test_results[f"pred_{t}"] = test_pred[:, i]
    test_results["uncertainty_rdot"] = mc_std_test

    pred_columns = [f"pred_{t}" for t in TARGET_COLUMNS]
    agg = {c: "mean" for c in pred_columns}
    agg["uncertainty_rdot"] = "mean"
    summary = test_results.groupby(FUEL_COLUMN, as_index=False).agg(agg)

    for t in TARGET_COLUMNS:
        summary[f"norm_{t}"] = minmax_norm(summary[f"pred_{t}"])

    wsum = sum(TARGET_WEIGHTS.values())
    summary["performance_score"] = (
        sum(TARGET_WEIGHTS[t] * summary[f"norm_{t}"] for t in TARGET_COLUMNS) / wsum
    )
    summary["uncertainty_norm"] = minmax_norm(summary["uncertainty_rdot"])
    summary["final_score"] = (
        summary["performance_score"]
        - UNCERTAINTY_PENALTY_K * summary["uncertainty_norm"]
    )
    summary = summary.sort_values("final_score", ascending=False).reset_index(drop=True)
    summary["rank"] = np.arange(1, len(summary) + 1)
    best = summary.iloc[0]

    print("\n========== FUEL RANKING (LABELED TEST SET) — LightGBM ==========")
    show = ["rank", FUEL_COLUMN] + pred_columns + ["uncertainty_rdot", "final_score"]
    show = [c for c in show if c in summary.columns]
    print(summary[show].to_string(index=False))

    print("\n========== BEST FUEL (LightGBM) ==========")
    print("Best Fuel               :", best[FUEL_COLUMN])
    print(
        "Regression Rate         : {:.4f} ± {:.4f} mm/s".format(
            float(best[PRED_RDOT]), float(best["uncertainty_rdot"])
        )
    )
    print("Pressure (pred)         : {:.4f}".format(float(best["pred_Pressure"])))
    print("Specific Impulse        : {:.2f}".format(float(best["pred_Isp"])))
    print("Thrust                  : {:.2f}".format(float(best["pred_Thrust"])))
    print("Characteristic Velocity : {:.2f}".format(float(best["pred_cstar"])))
    print("Final Score             : {:.4f}".format(float(best["final_score"])))

    plot_gox_vs_rdot(results, gox_col, sheet_names)
    plot_pressure_vs_rdot(results, pressure_col, sheet_names)

    with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
        results.to_excel(writer, sheet_name="All_Predictions", index=False)
        test_results.to_excel(writer, sheet_name="Labeled_Test_Predictions", index=False)
        summary.to_excel(writer, sheet_name="Fuel_Ranking", index=False)

    lines = [
        "HYBRID ROCKET FUEL BLEND — ML SUMMARY (LightGBM)",
        "=" * 60,
        f"Main model     : LightGBM",
        f"Excel file     : {EXCEL_PATH}",
        f"Sheets used    : {sheet_names}",
        f"Input features : {feature_cols}",
        f"Predicted targets : {TARGET_COLUMNS}",
        f"Total input rows : {len(data)}",
        f"Labeled rows     : {int(labeled_mask.sum())}",
        f"Train / Test (labeled) : {len(x_train)} / {len(x_test)}",
        "",
        "PER-TARGET TEST METRICS (labeled test)",
        "-" * 60,
    ]
    for i, t in enumerate(TARGET_COLUMNS):
        lines.append(
            f"{t:<20}  Test R2={test_r2[i]:.4f}  MAE={test_mae[i]:.4f}  RMSE={test_rmse[i]:.4f}"
        )
    lines += [
        "",
        f"MC Dropout TEST R2={mc_r2:.4f}  MAE={mc_mae:.4f}  "
        f"mean sigma={mc_std_test.mean():.4f}",
        "",
        "FUEL RANKING (LABELED TEST SET)",
        "-" * 60,
        summary[show].to_string(index=False),
        "",
        "BEST FUEL",
        "-" * 60,
        f"Best Fuel               : {best[FUEL_COLUMN]}",
        f"Regression Rate         : {float(best[PRED_RDOT]):.4f} ± "
        f"{float(best['uncertainty_rdot']):.4f} mm/s",
        f"Pressure (pred)         : {float(best['pred_Pressure']):.4f}",
        f"Specific Impulse        : {float(best['pred_Isp']):.2f}",
        f"Thrust                  : {float(best['pred_Thrust']):.2f}",
        f"Characteristic Velocity : {float(best['pred_cstar']):.2f}",
        f"Final Score             : {float(best['final_score']):.4f}",
        "",
        f"Excel : {OUTPUT_EXCEL}",
        f"Graph : {OUTPUT_GRAPH1}",
    ]
    OUTPUT_TXT.write_text("\n".join(lines), encoding="utf-8")

    print(f"\nSummary: {OUTPUT_TXT.resolve()}")
    print(f"Excel  : {OUTPUT_EXCEL.resolve()}")


if __name__ == "__main__":
    main()


Sheets used      : ['Firing 01', 'Firing 02', 'Firing 03', 'Firing 04']
Total input rows : 376
Labeled rows     : 376  (all targets filled — used for training)
Unlabeled rows   : 0  (inputs only — prediction only)
Gox column       : Gox
Targets          : ['Pressure', 'rdot_mm_s', 'Isp', 'Thrust', 'cstar']
Pred rdot column : pred_rdot_mm_s
Main model       : LightGBM
Train (labeled): 300 | Test (labeled): 76

Training Multi-Output LightGBM on labeled data...

Per-target performance (labeled test set) — LightGBM:
  Pressure             Train R2=0.000 | Test R2=-0.001 | gap=0.001 | MAE=0.0263 | RMSE=0.0376
  rdot_mm_s            Train R2=0.957 | Test R2=0.941 | gap=0.016 | MAE=0.0081 | RMSE=0.0246
  Isp                  Train R2=0.997 | Test R2=0.985 | gap=0.011 | MAE=0.1107 | RMSE=0.2226
  Thrust               Train R2=0.998 | Test R2=0.991 | gap=0.007 | MAE=0.1534 | RMSE=0.3073
  cstar                Train R2=0.000 | Test R2=-0.001 | gap=0.001 | MAE=0.0001 | RMSE=0.0001

Training MC Dr